# A CNN, One Step at a Time — PyTorch Version

This notebook takes **one photo** and pushes it through every stage of a convolutional neural network
by hand, so you can *see* what each stage does to the image:

```
photo -> convolution -> ReLU -> max pooling -> flatten -> fully connected layer
```

It's a PyTorch conversion of the original TensorFlow notebook. The steps are identical; only the
library calls change.

### The four differences between TensorFlow and PyTorch you'll meet here

| | TensorFlow | PyTorch |
|---|---|---|
| **Image layout** | channels **last**: `(batch, height, width, channels)` | channels **first**: `(batch, channels, height, width)` |
| **Filter (kernel) layout** | `(height, width, in_channels, out_channels)` | `(out_channels, in_channels, height, width)` |
| **Fully connected layer** | `Dense(64)` — works out the input size for you | `nn.Linear(in_features, 64)` — you give the input size |
| **Where the activation goes** | inside the layer: `Dense(64, activation='relu')` | a separate step: `nn.ReLU()` or `F.relu(...)` |

The first one causes the most confusion. Both libraries hold exactly the same numbers; they just
disagree about **which slot the colour channel goes in**. Think of it as writing a date as
day/month/year versus month/day/year: same date, different order, and mixing them up gives nonsense.

In [ ]:
# ==========================================
# SETUP
# ==========================================
import os
import torch
import torch.nn as nn
import torch.nn.functional as F                     # the 'functional' versions: F.conv2d, F.relu, ...
from torchvision.io import decode_image, ImageReadMode
from torchvision.transforms.v2 import functional as TF   # image resizing and type conversion
import matplotlib.pyplot as plt

plt.rc('image', cmap='gray')                        # show single-channel images in greyscale
plt.rc('figure', autolayout=True)

torch.manual_seed(42)                               # so the random Linear layer at the end is repeatable

## Step 1 — Load the photo in greyscale

Three jobs, same as the TensorFlow version:

1. **Read** the file as a single greyscale channel
2. **Resize** it to 300×300
3. **Convert** the pixels from whole numbers (0–255) to decimals (0.0–1.0)

> **A bug this fixes.** In the TensorFlow original, `tf.image.resize` already hands back decimals, so
> the `convert_image_dtype(..., tf.float32)` line after it silently did nothing — the pixels stayed in
> the 0–255 range. That's why the original printed values like `135.` and `81.` at the end. Here the
> pixels really do end up between 0 and 1, so you should see values around `0.53` and `0.32`
> instead: **the same numbers, divided by 255.**

In [ ]:
# ==========================================
# STEP 1: LOAD, RESIZE, CONVERT
# ==========================================
image_path = "output.png"
if not os.path.exists(image_path):
    raise FileNotFoundError(
        f"Can't find {image_path}. Put it in the same folder as this notebook "
        "(in Google Colab: open the Files panel on the left and upload it)."
    )

# Read as ONE greyscale channel. PyTorch gives back (channels, height, width) -- channels FIRST
image = decode_image(image_path, mode=ImageReadMode.GRAY)          # whole numbers 0-255

# Whole numbers 0-255  ->  decimals 0.0-1.0 (scale=True does the dividing by 255)
image = TF.to_dtype(image, torch.float32, scale=True)

# Resize to 300x300. Bilinear, no antialiasing -- the same defaults tf.image.resize uses
image = TF.resize(image, [300, 300], interpolation=TF.InterpolationMode.BILINEAR, antialias=False)

print("Original Image Shape:", tuple(image.shape), " <- (channels, height, width)")
print("TensorFlow showed      (300, 300, 1)            <- (height, width, channels): same image, other order")
print(f"pixel range: {image.min():.2f} to {image.max():.2f}")

plt.figure(figsize=(5, 5))
plt.imshow(image.squeeze())             # squeeze drops the size-1 channel so matplotlib gets a plain 2D grid
plt.title("Original Image")
plt.axis('off')
plt.show()

## Step 2 — Add a batch dimension

Layers are built to process a **batch** of images at once, even when the batch holds only one. So we
wrap our single image in a batch of size 1.

- TensorFlow added it with `tf.expand_dims(image, axis=0)` -> `(1, 300, 300, 1)`
- PyTorch uses `unsqueeze(0)` -> `(1, 1, 300, 300)`

In [ ]:
# ==========================================
# STEP 2: A BATCH OF ONE
# ==========================================
image = image.unsqueeze(0)              # (1, 300, 300) -> (1, 1, 300, 300)

print("With batch dimension:", tuple(image.shape), " <- (batch, channels, height, width)")

## Step 3 — The filter: an edge detector

A **kernel** (or filter) is a small grid of numbers slid across the image. This particular one is a
classic **edge detector**:

```
-1 -1 -1
-1  8 -1
-1 -1 -1
```

Read it as: *"take 8 times this pixel, then subtract each of its 8 neighbours."*

- In a **flat area**, every neighbour equals the centre, so `8×centre − 8×centre = 0`. **Black.**
- On an **edge**, the centre differs from its neighbours, so the result is far from zero. **Bright.**

A trained CNN learns filters like this by itself. Here we just write one by hand to see what a filter
does.

**The shape is the part that changes.** TensorFlow wants the kernel as
`(height, width, in_channels, out_channels)` = `(3, 3, 1, 1)`. PyTorch wants
`(out_channels, in_channels, height, width)` = `(1, 1, 3, 3)`.

In [ ]:
# ==========================================
# STEP 3: BUILD THE EDGE-DETECTING KERNEL
# ==========================================
kernel = torch.tensor([
    [-1, -1, -1],
    [-1,  8, -1],
    [-1, -1, -1],
], dtype=torch.float32)

# TensorFlow: tf.reshape(kernel, [3, 3, 1, 1])   -> (height, width, in, out)
# PyTorch   : (out_channels, in_channels, height, width)
kernel = kernel.reshape(1, 1, 3, 3)      # 1 filter out, 1 channel in, 3x3 window

print("Kernel shape:", tuple(kernel.shape), " <- (out_channels, in_channels, height, width)")

## Step 4 — Convolution: slide the filter over the whole image

`F.conv2d` slides the 3×3 kernel to every position in the image and writes down the result at each
stop.

`padding='same'` adds a border of zeros around the image so the output stays 300×300 — the same size
as the input. (Without it, the edges would shrink by one pixel on every side.)

In [ ]:
# ==========================================
# STEP 4: CONVOLUTION
# ==========================================
# TensorFlow: tf.nn.conv2d(input=image, filters=kernel, strides=1, padding='SAME')
conv_output = F.conv2d(
    input=image,          # (1, 1, 300, 300)
    weight=kernel,        # (1, 1, 3, 3)   <- PyTorch calls the filter 'weight'
    stride=1,             # move one pixel at a time
    padding='same',       # zero border, so the output stays 300x300
)

print("After Convolution Shape:", tuple(conv_output.shape))

plt.figure(figsize=(5, 5))
plt.imshow(conv_output.squeeze())
plt.title("After Convolution")
plt.axis('off')
plt.show()

## Step 5 — ReLU: keep the positives, zero the negatives

The edge detector produces both positive and negative numbers. **ReLU** keeps every positive value and
replaces every negative with 0:

```
if the number is negative -> 0
otherwise                 -> leave it alone
```

It's the simplest possible "bend" in a network, and it's what lets stacked layers learn more than a
straight line could.

In [ ]:
# ==========================================
# STEP 5: ReLU
# ==========================================
# TensorFlow: tf.nn.relu(conv_output)
relu_output = F.relu(conv_output)

print("After ReLU Shape:", tuple(relu_output.shape), " <- ReLU never changes the shape")
print(f"smallest value before ReLU: {conv_output.min():.3f}   after: {relu_output.min():.3f}")

plt.figure(figsize=(5, 5))
plt.imshow(relu_output.squeeze())
plt.title("After ReLU")
plt.axis('off')
plt.show()

## Step 6 — Max pooling: zoom out

**Max pooling** cuts the image into 2×2 squares and keeps only the **biggest** value from each one. The
height and width halve: 300×300 becomes 150×150.

Keeping the biggest value asks *"was there a strong edge anywhere in this little square?"* — which keeps
the important information while throwing away three quarters of the numbers.

> PyTorch's `max_pool2d` has no `padding='same'` option. It isn't needed here: 300 divides evenly by 2,
> so TensorFlow's `'SAME'` and PyTorch's default give the identical 150×150 result.

In [ ]:
# ==========================================
# STEP 6: MAX POOLING
# ==========================================
# TensorFlow: tf.nn.max_pool2d(input=relu_output, ksize=2, strides=2, padding='SAME')
pool_output = F.max_pool2d(
    relu_output,
    kernel_size=2,        # look at 2x2 squares
    stride=2,             # jump 2 pixels each time, so the squares don't overlap
)

print("After Pooling Shape:", tuple(pool_output.shape), " <- height and width halved")

plt.figure(figsize=(5, 5))
plt.imshow(pool_output.squeeze())
plt.title("After Max Pooling")
plt.axis('off')
plt.show()

## Step 7 — Flatten: unroll the grid into one long row

A fully connected layer can't take a grid — it wants a flat list of numbers. **Flatten** unrolls the
150×150 grid into one row of 150 × 150 = **22,500** numbers, keeping the batch dimension separate.

### Why the first 20 values look like that

They're the very top row of the image, and they come from **the zero border that `padding='same'`
added**, not from anything in the photo.

The TensorFlow version printed `135` followed by a run of `81`s. That pattern gives it away. Away from
the border, the edge detector turns any flat area into 0. But along the border, some of a pixel's
neighbours are padding zeros instead of real pixels, so the sum no longer cancels:

- on the **top edge**, 3 of the 8 neighbours are padding, leaving `3 × pixel`
- in the **corner**, 5 of the 8 are padding, leaving `5 × pixel`

`135 = 5 × 27` and `81 = 3 × 27`: the top of the photo is a flat dark background with a brightness of
27 out of 255. On the 0-to-1 scale that's 0.106, so here you should get about **0.53 and 0.32** —
and multiplying by 255 (the second printout) takes you back to about **135 and 81**. If a pixel or two
decodes one shade differently from TensorFlow's JPEG reader, the values shift very slightly; the
pattern is what matters.

The takeaway: the bright "edges" at the very top aren't edges in the picture at all. They're the
filter reacting to the padding.

In [ ]:
# ==========================================
# STEP 7: FLATTEN
# ==========================================
# TensorFlow: tf.keras.layers.Flatten()
flatten_layer = nn.Flatten()               # keeps dim 0 (the batch), flattens everything after it
flatten_output = flatten_layer(pool_output)

print("After Flatten Shape:", tuple(flatten_output.shape), f" <- 1 x 150 x 150 = {1 * 150 * 150:,}")

print("\nFirst 20 values of Flattened Vector:\n")
print(flatten_output[0, :20])

print("\nThe same values x 255 (to compare with the TensorFlow version's 0-255 numbers):\n")
print((flatten_output[0, :20] * 255).round())

## Step 8 — The fully connected (dense) layer

A **fully connected** layer connects every one of the 22,500 inputs to each of its 64 outputs.

Two differences from Keras' `Dense(units=64, activation='relu')`:

1. **You tell PyTorch the input size.** Keras works out "22,500 inputs" the first time data flows
   through. `nn.Linear` needs it up front: `nn.Linear(22500, 64)`. Below we read it straight off the
   flattened tensor rather than typing 22500, so it can never be wrong. (`nn.LazyLinear(64)` would also
   work it out for you, like Keras.)
2. **The activation is a separate step.** `Linear` does the multiply-and-add; `F.relu` is applied
   afterwards.

The layer's weights start random and we haven't trained anything, so these 64 numbers don't *mean*
anything yet. The point is the shape: 22,500 numbers in, 64 out.

In [ ]:
# ==========================================
# STEP 8: FULLY CONNECTED LAYER
# ==========================================
# TensorFlow: tf.keras.layers.Dense(units=64, activation='relu')
dense_layer = nn.Linear(
    in_features=flatten_output.shape[1],    # 22,500 -- read from the data, never hard-coded
    out_features=64,
)
dense_output = F.relu(dense_layer(flatten_output))   # multiply-and-add, THEN ReLU

print("After Fully Connected Layer Shape:", tuple(dense_output.shape))
print(f"this one layer has {sum(p.numel() for p in dense_layer.parameters()):,} weights  "
      "(22,500 x 64, plus 64 biases)")

## Bonus — the whole pipeline as one PyTorch model

Everything above was done step by step to see inside. In real PyTorch code the same stages are packed
into one model with `nn.Sequential`, and the image goes through with a single call.

`nn.Conv2d` normally **learns** its filter during training. Here we copy our hand-made edge detector
into it, so the result should match the step-by-step version.

We run it inside `torch.no_grad()`, as you should whenever you're only *using* a model rather than
training it (Phase 2). As a bonus, that makes the match exact: when PyTorch is tracking gradients it can
pick a slightly different convolution routine, whose rounding differs in the seventh decimal place.

In [ ]:
# ==========================================
# BONUS: THE SAME PIPELINE AS ONE MODEL
# ==========================================
model = nn.Sequential(
    nn.Conv2d(in_channels=1, out_channels=1, kernel_size=3, padding='same', bias=False),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2, stride=2),
    nn.Flatten(),
)

with torch.no_grad():
    model[0].weight.copy_(kernel)            # put our edge detector into the conv layer

with torch.no_grad():                        # only predicting, not training
    model_output = model(image)              # all four stages in one call

print("model output shape:", tuple(model_output.shape))
print("largest difference from the step-by-step version:", (model_output - flatten_output).abs().max().item())
print("same result?", torch.allclose(model_output, flatten_output))